# MVRV Exit Fix: Actually Using the Signal

**Problem:** MVRV > 3.0 only triggers 2 times in 7 years!
- Stop loss doing all the work (5 exits)
- Max hold timing out (3 exits)
- MVRV almost never fires

**Solutions to test:**
1. Lower MVRV threshold (2.0, 2.25, 2.5)
2. MVRV triggers trailing stop (not hard exit)
3. Tiered exit (scale out as MVRV rises)

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Let's actually USE the MVRV signal! 🎯")

In [ ]:
# Load data
DATA_DIR = Path("../data/daily")

sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")

df = sopr.join(sopr_sth, how='inner').join(price, how='inner').join(mvrv, how='inner')
df = df.sort_index()
df = df[df.index >= '2018-12-15']

close = df['price']
print(f"Data: {len(df)} rows")
print(f"MVRV range: {df['mvrv'].min():.2f} to {df['mvrv'].max():.2f}")

In [ ]:
# How often is MVRV above various thresholds?
print("MVRV THRESHOLD FREQUENCY")
print("="*50)
for thresh in [1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.5]:
    days_above = (df['mvrv'] > thresh).sum()
    pct = days_above / len(df) * 100
    print(f"MVRV > {thresh}: {days_above} days ({pct:.1f}%)")

In [ ]:
# Visualize MVRV distribution and thresholds
fig = make_subplots(rows=2, cols=1, row_heights=[0.6, 0.4],
                    subplot_titles=['MVRV Over Time', 'MVRV Distribution'])

fig.add_trace(go.Scatter(x=df.index, y=df['mvrv'], name='MVRV'), row=1, col=1)
for thresh, color in [(2.0, 'green'), (2.5, 'orange'), (3.0, 'red')]:
    fig.add_hline(y=thresh, line_dash='dash', line_color=color, row=1, col=1,
                  annotation_text=f'MVRV {thresh}')

fig.add_trace(go.Histogram(x=df['mvrv'], nbinsx=50, name='Distribution'), row=2, col=1)
for thresh, color in [(2.0, 'green'), (2.5, 'orange'), (3.0, 'red')]:
    fig.add_vline(x=thresh, line_dash='dash', line_color=color, row=2, col=1)

fig.update_layout(height=600, title_text='MVRV Thresholds - 3.0 is RARE!')
fig.show()

In [ ]:
# Entry signal
both_below_1 = (df['sopr'] < 1) & (df['sopr_sth'] < 1)
entries = both_below_1 & ~both_below_1.shift(1).fillna(False)
print(f"Entry signals: {entries.sum()}")

---
## Strategy 1: Lower MVRV Threshold

In [ ]:
def backtest_mvrv_exit(df, entries, exit_mvrv=2.5, stop_loss=0.20, max_hold_days=365):
    """Basic MVRV exit backtest."""
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            pnl = (current_price - entry_price) / entry_price
            
            # Stop loss
            if stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            # MVRV exit
            if current_mvrv >= exit_mvrv:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'mvrv_exit'
                break
            
            # Max hold
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

# Test different thresholds
print("MVRV THRESHOLD COMPARISON")
print("="*100)
print(f"{'Threshold':<12} {'Trades':>8} {'Return':>10} {'Win%':>8} {'MVRV Exits':>12} {'Stop Exits':>12} {'MaxHold':>10}")
print("-"*100)

for thresh in [1.75, 2.0, 2.25, 2.5, 2.75, 3.0]:
    trades = backtest_mvrv_exit(df, entries, exit_mvrv=thresh, stop_loss=0.20)
    
    total_ret = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    mvrv_exits = (trades['exit_reason'] == 'mvrv_exit').sum()
    stop_exits = (trades['exit_reason'] == 'stop_loss').sum()
    max_exits = (trades['exit_reason'] == 'max_hold').sum()
    
    print(f"MVRV > {thresh:<5} {len(trades):>8} {total_ret*100:>9.0f}% {win_rate*100:>7.0f}% "
          f"{mvrv_exits:>12} {stop_exits:>12} {max_exits:>10}")

---
## Strategy 2: MVRV Triggers Trailing Stop

In [ ]:
def backtest_mvrv_trailing(
    df, entries,
    mvrv_trail_trigger=2.0,  # MVRV level that activates trailing stop
    trailing_pct=0.15,       # Trail amount once activated
    stop_loss=0.20,          # Initial stop loss
    max_hold_days=365
):
    """
    MVRV doesn't exit directly - it ACTIVATES a trailing stop.
    This lets profits run while MVRV is high.
    """
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        peak_price = entry_price
        trailing_active = False
        trail_activated_at = None
        
        exit_date = None
        exit_price = None
        exit_reason = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            # Update peak
            if current_price > peak_price:
                peak_price = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            # Activate trailing stop when MVRV hits threshold
            if not trailing_active and current_mvrv >= mvrv_trail_trigger:
                trailing_active = True
                trail_activated_at = current_mvrv
            
            # Check trailing stop if active
            if trailing_active:
                trail_stop = peak_price * (1 - trailing_pct)
                if current_price <= trail_stop:
                    exit_date = current_date
                    exit_price = trail_stop
                    exit_reason = 'mvrv_trailing'
                    break
            
            # Check initial stop loss (before MVRV activates trailing)
            if not trailing_active and stop_loss and pnl <= -stop_loss:
                exit_date = current_date
                exit_price = entry_price * (1 - stop_loss)
                exit_reason = 'stop_loss'
                break
            
            # Max hold
            if days_held >= max_hold_days:
                exit_date = current_date
                exit_price = current_price
                exit_reason = 'max_hold'
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
            exit_reason = 'end_of_data'
        
        pnl = (exit_price - entry_price) / entry_price
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': exit_date,
            'exit_price': exit_price,
            'pnl_pct': pnl,
            'days_held': (exit_date - entry_date).days,
            'exit_reason': exit_reason,
            'peak_price': peak_price,
            'trail_activated': trailing_active
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test MVRV trailing configurations
configs = [
    {'trigger': 1.75, 'trail': 0.15},
    {'trigger': 1.75, 'trail': 0.20},
    {'trigger': 2.0, 'trail': 0.15},
    {'trigger': 2.0, 'trail': 0.20},
    {'trigger': 2.0, 'trail': 0.25},
    {'trigger': 2.25, 'trail': 0.15},
    {'trigger': 2.25, 'trail': 0.20},
    {'trigger': 2.5, 'trail': 0.20},
]

print("MVRV TRAILING STOP COMPARISON")
print("="*110)
print(f"{'Config':<25} {'Trades':>8} {'Return':>10} {'Win%':>8} {'Trail Exits':>12} {'Stop Exits':>12} {'AvgDays':>10}")
print("-"*110)

trailing_results = []

for cfg in configs:
    trades = backtest_mvrv_trailing(
        df, entries,
        mvrv_trail_trigger=cfg['trigger'],
        trailing_pct=cfg['trail'],
        stop_loss=0.20
    )
    
    total_ret = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    trail_exits = (trades['exit_reason'] == 'mvrv_trailing').sum()
    stop_exits = (trades['exit_reason'] == 'stop_loss').sum()
    avg_days = trades['days_held'].mean()
    
    name = f"MVRV>{cfg['trigger']} → {cfg['trail']*100:.0f}% trail"
    print(f"{name:<25} {len(trades):>8} {total_ret*100:>9.0f}% {win_rate*100:>7.0f}% "
          f"{trail_exits:>12} {stop_exits:>12} {avg_days:>10.0f}")
    
    trailing_results.append({
        'config': name,
        'trigger': cfg['trigger'],
        'trail': cfg['trail'],
        'total_return': total_ret,
        'win_rate': win_rate,
        'trail_exits': trail_exits,
        'trades': trades
    })

---
## Strategy 3: Tiered Exit (Scale Out)

In [ ]:
def backtest_tiered_exit(
    df, entries,
    tier1_mvrv=2.0,   # Exit 33% at this level
    tier2_mvrv=2.5,   # Exit another 33%
    tier3_mvrv=3.0,   # Exit final 33%
    stop_loss=0.20,
    max_hold_days=365
):
    """
    Tiered exit - scale out as MVRV rises.
    Simulates selling 1/3 at each tier.
    Returns weighted average PnL.
    """
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        
        # Track tier exits
        tier1_exit = None
        tier2_exit = None
        tier3_exit = None
        final_exit = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            current_mvrv = df['mvrv'].iloc[j]
            days_held = j - entry_idx
            
            pnl = (current_price - entry_price) / entry_price
            
            # Stop loss (affects remaining position)
            if stop_loss and pnl <= -stop_loss:
                stop_price = entry_price * (1 - stop_loss)
                if tier1_exit is None:
                    tier1_exit = stop_price
                if tier2_exit is None:
                    tier2_exit = stop_price
                if tier3_exit is None:
                    tier3_exit = stop_price
                final_exit = ('stop_loss', current_date)
                break
            
            # Tier exits
            if tier1_exit is None and current_mvrv >= tier1_mvrv:
                tier1_exit = current_price
            if tier2_exit is None and current_mvrv >= tier2_mvrv:
                tier2_exit = current_price
            if tier3_exit is None and current_mvrv >= tier3_mvrv:
                tier3_exit = current_price
                final_exit = ('tier3_complete', current_date)
                break
            
            # Max hold
            if days_held >= max_hold_days:
                if tier1_exit is None:
                    tier1_exit = current_price
                if tier2_exit is None:
                    tier2_exit = current_price
                if tier3_exit is None:
                    tier3_exit = current_price
                final_exit = ('max_hold', current_date)
                break
        
        if final_exit is None:
            current_price = close.iloc[-1]
            if tier1_exit is None:
                tier1_exit = current_price
            if tier2_exit is None:
                tier2_exit = current_price
            if tier3_exit is None:
                tier3_exit = current_price
            final_exit = ('end_of_data', df.index[-1])
        
        # Calculate weighted average exit
        avg_exit = (tier1_exit + tier2_exit + tier3_exit) / 3
        pnl = (avg_exit - entry_price) / entry_price
        
        trades.append({
            'entry_date': entry_date,
            'entry_price': entry_price,
            'exit_date': final_exit[1],
            'tier1_exit': tier1_exit,
            'tier2_exit': tier2_exit,
            'tier3_exit': tier3_exit,
            'avg_exit': avg_exit,
            'pnl_pct': pnl,
            'days_held': (final_exit[1] - entry_date).days,
            'exit_reason': final_exit[0]
        })
        
        while i < len(entry_indices) and entry_indices[i] <= final_exit[1]:
            i += 1
    
    return pd.DataFrame(trades)

In [ ]:
# Test tiered configurations
tiered_configs = [
    {'t1': 1.75, 't2': 2.25, 't3': 2.75},
    {'t1': 1.75, 't2': 2.0, 't3': 2.5},
    {'t1': 2.0, 't2': 2.5, 't3': 3.0},
    {'t1': 2.0, 't2': 2.25, 't3': 2.5},
    {'t1': 1.5, 't2': 2.0, 't3': 2.5},
]

print("\n\nTIERED EXIT COMPARISON")
print("="*100)
print(f"{'Tiers':<25} {'Trades':>8} {'Return':>10} {'Win%':>8} {'AvgDays':>10}")
print("-"*100)

tiered_results = []

for cfg in tiered_configs:
    trades = backtest_tiered_exit(
        df, entries,
        tier1_mvrv=cfg['t1'],
        tier2_mvrv=cfg['t2'],
        tier3_mvrv=cfg['t3'],
        stop_loss=0.20
    )
    
    total_ret = (1 + trades['pnl_pct']).prod() - 1
    win_rate = (trades['pnl_pct'] > 0).mean()
    avg_days = trades['days_held'].mean()
    
    name = f"{cfg['t1']}/{cfg['t2']}/{cfg['t3']}"
    print(f"{name:<25} {len(trades):>8} {total_ret*100:>9.0f}% {win_rate*100:>7.0f}% {avg_days:>10.0f}")
    
    tiered_results.append({
        'config': name,
        'total_return': total_ret,
        'win_rate': win_rate,
        'trades': trades
    })

---
## Walk-Forward Validation

In [ ]:
def walk_forward(df, entries, backtest_func, **kwargs):
    """Generic walk-forward."""
    results = []
    close = df['price']
    
    train_days = 365
    test_days = 90
    step_days = 90
    
    total_days = len(df)
    n_folds = (total_days - train_days) // step_days
    
    for fold in range(n_folds):
        test_start = train_days + fold * step_days
        test_end = min(test_start + test_days, total_days)
        
        test_df = df.iloc[test_start:test_end]
        test_entries = entries.iloc[test_start:test_end]
        test_close = close.iloc[test_start:test_end]
        
        trades = backtest_func(test_df, test_entries, **kwargs)
        
        strat_return = (1 + trades['pnl_pct']).prod() - 1 if len(trades) > 0 else 0
        hold_return = (test_close.iloc[-1] / test_close.iloc[0]) - 1
        
        results.append({
            'strat_return': strat_return,
            'hold_return': hold_return,
            'beat_hold': strat_return > hold_return
        })
    
    wf_df = pd.DataFrame(results)
    return wf_df['beat_hold'].mean(), (wf_df['strat_return'] - wf_df['hold_return']).mean()

In [ ]:
# Walk-forward all strategies
print("\n\nWALK-FORWARD VALIDATION")
print("="*80)
print(f"{'Strategy':<40} {'Beat Rate':>15} {'Avg Excess':>15}")
print("-"*80)

# Baseline: Original trailing stop
def baseline_trailing(df, entries, stop_loss=0.08, trailing=0.12, min_profit=0.05, max_hold=180):
    trades = []
    entry_indices = entries[entries].index.tolist()
    close = df['price']
    
    i = 0
    while i < len(entry_indices):
        entry_date = entry_indices[i]
        entry_idx = df.index.get_loc(entry_date)
        entry_price = close.iloc[entry_idx]
        peak = entry_price
        is_trailing = False
        current_stop = entry_price * (1 - stop_loss)
        
        exit_date = exit_price = None
        
        for j in range(entry_idx + 1, len(df)):
            current_date = df.index[j]
            current_price = close.iloc[j]
            days = j - entry_idx
            
            if current_price > peak:
                peak = current_price
            
            pnl = (current_price - entry_price) / entry_price
            
            if not is_trailing and pnl >= min_profit:
                is_trailing = True
            
            if is_trailing:
                trail = peak * (1 - trailing)
                if trail > current_stop:
                    current_stop = trail
            
            if current_price <= current_stop:
                exit_date = current_date
                exit_price = current_stop
                break
            
            if days >= max_hold:
                exit_date = current_date
                exit_price = current_price
                break
        
        if exit_date is None:
            exit_date = df.index[-1]
            exit_price = close.iloc[-1]
        
        trades.append({'pnl_pct': (exit_price - entry_price) / entry_price})
        
        while i < len(entry_indices) and entry_indices[i] <= exit_date:
            i += 1
    
    return pd.DataFrame(trades)

# Test all
wf_results = []

# Baseline
beat, excess = walk_forward(df, entries, baseline_trailing)
print(f"{'Baseline (8%SL/12%Trail)':<40} {beat*100:>14.0f}% {excess*100:>+14.1f}%")
wf_results.append({'name': 'Baseline Trail', 'beat': beat, 'excess': excess})

# MVRV hard exits
for thresh in [2.0, 2.25, 2.5]:
    beat, excess = walk_forward(df, entries, backtest_mvrv_exit, exit_mvrv=thresh, stop_loss=0.20)
    print(f"{f'MVRV > {thresh} (hard exit)':<40} {beat*100:>14.0f}% {excess*100:>+14.1f}%")
    wf_results.append({'name': f'MVRV>{thresh} hard', 'beat': beat, 'excess': excess})

# MVRV trailing
for trigger, trail in [(1.75, 0.15), (2.0, 0.15), (2.0, 0.20), (2.25, 0.20)]:
    beat, excess = walk_forward(df, entries, backtest_mvrv_trailing, 
                                 mvrv_trail_trigger=trigger, trailing_pct=trail, stop_loss=0.20)
    print(f"{f'MVRV>{trigger} → {trail*100:.0f}% trail':<40} {beat*100:>14.0f}% {excess*100:>+14.1f}%")
    wf_results.append({'name': f'MVRV>{trigger}→{trail*100:.0f}%trail', 'beat': beat, 'excess': excess})

# Tiered
for cfg in tiered_configs[:3]:
    beat, excess = walk_forward(df, entries, backtest_tiered_exit,
                                 tier1_mvrv=cfg['t1'], tier2_mvrv=cfg['t2'], tier3_mvrv=cfg['t3'], stop_loss=0.20)
    name = f"Tiered {cfg['t1']}/{cfg['t2']}/{cfg['t3']}"
    print(f"{name:<40} {beat*100:>14.0f}% {excess*100:>+14.1f}%")
    wf_results.append({'name': name, 'beat': beat, 'excess': excess})

In [ ]:
# Visualize results
wf_df = pd.DataFrame(wf_results).sort_values('beat', ascending=False)

fig = go.Figure()

colors = ['green' if x > 0.60 else 'orange' if x > 0.55 else 'gray' for x in wf_df['beat']]

fig.add_trace(go.Bar(
    x=wf_df['name'],
    y=wf_df['beat'] * 100,
    marker_color=colors,
    text=[f"{x:.0f}%" for x in wf_df['beat']*100],
    textposition='outside'
))

fig.add_hline(y=50, line_dash='dash', line_color='red')
fig.add_hline(y=54, line_dash='dot', line_color='orange', annotation_text='54% baseline')

fig.update_layout(
    title='Walk-Forward Beat Rate - All Strategies',
    yaxis_title='Beat Buy & Hold %',
    xaxis_tickangle=-45,
    height=500
)
fig.show()

---
## Best Strategy Trade Details

In [ ]:
# Find best and show trades
best = max(wf_results, key=lambda x: x['beat'])
print(f"\n\n🏆 BEST STRATEGY: {best['name']}")
print(f"   Beat Rate: {best['beat']*100:.0f}%")
print(f"   Avg Excess: {best['excess']*100:+.1f}%")

In [ ]:
# Summary
print("\n" + "="*80)
print("SUMMARY: MVRV EXIT FIX")
print("="*80)

print(f"\n❌ PROBLEM: MVRV > 3.0 rarely fires (only 2 exits in 7 years)")
print(f"\n✅ SOLUTIONS TESTED:")
print(f"   1. Lower threshold (MVRV > 2.0, 2.25, 2.5)")
print(f"   2. MVRV triggers trailing stop (not hard exit)")
print(f"   3. Tiered exit (scale out at multiple levels)")

print(f"\n🏆 BEST: {best['name']}")
print(f"   Beat Rate: {best['beat']*100:.0f}% (vs 54% baseline, 62% original MVRV>3)")

print("\n" + "="*80)

In [ ]:
# Save results
import json

results = {
    'problem': 'MVRV > 3.0 only fires 2 times in 7 years',
    'solutions_tested': ['lower_threshold', 'mvrv_trailing', 'tiered_exit'],
    'walk_forward_results': wf_results,
    'best_strategy': best
}

with open('../data/mvrv_exit_fix_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print("Saved to ../data/mvrv_exit_fix_results.json")